# Chat with PDF: простий RAG без векторної бази

**Тема:** Retrieval-Augmented Generation для PDF-документа

RAG pipeline:

```text
PDF -> текст -> chunks -> embeddings -> cosine similarity -> top-3 chunks -> prompt -> LLM -> відповідь
```

## Що потрібно зробити

Заповніть клітинки з позначкою `TODO`. Після завершення notebook має вміти:

1. читати текст із PDF;
2. розбивати текст на частини з overlap;
3. будувати embeddings для chunks через `SentenceTransformer`;
4. знаходити top-k релевантних chunks за cosine similarity;
5. формувати prompt із контекстом;
6. отримувати відповідь від LLM або підготувати prompt для доступної моделі;
7. перевіряти якість відповіді на 5 питаннях.

Якщо у вас є власний PDF, замініть шлях у клітинці `PDF_PATH`. Якщо ні, використайте тестовий файл `data/sample_washing_machine_manual.pdf`.

**Очікуваний результат:** після виконання всіх TODO notebook читає PDF, ділить текст на chunks, будує embeddings, знаходить релевантні фрагменти, формує RAG prompt, за потреби викликає Gemini API та дає відповіді на тестові питання.

## Встановлення бібліотек

In [1]:
# !pip -q install pypdf sentence-transformers requests

## 1. Імпорти та налаштування

In [ ]:
from pathlib import Path
import os
import re

import numpy as np
import pandas as pd
import requests
from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 160)

In [ ]:
PDF_PATH = Path("data/sample_washing_machine_manual.pdf")

# Якщо працюєте з власним PDF, покладіть файл поруч із notebook або в папку data
# і замініть шлях, наприклад:
# PDF_PATH = Path("data/my_document.pdf")

print("PDF exists:", PDF_PATH.exists())
print("PDF path:", PDF_PATH)

## 2. Читання тексту з PDF

Заповніть функцію `load_pdf_text`.

Підказки:

- створіть `PdfReader(pdf_path)`;
- пройдіться по `reader.pages`;
- для кожної сторінки викличте `page.extract_text()`;
- якщо текст сторінки порожній, підставте порожній рядок;
- об'єднайте сторінки через `"
"`.

In [ ]:
def load_pdf_text(pdf_path):
    """Повертає весь текст із PDF одним рядком."""
    pdf_path = Path(pdf_path)

    # TODO: створіть PdfReader для pdf_path
    # TODO: дістаньте текст із кожної сторінки
    # TODO: об'єднайте сторінки в один рядок

    raise NotImplementedError("Заповніть функцію load_pdf_text")

**Очікуваний результат:** функція `load_pdf_text(PDF_PATH)` повертає непорожній рядок із текстом PDF.

У наступній комірці мають вивестися:
- кількість символів у документі, більша за 0;
- перші 500 символів тексту без помилки `NotImplementedError`.

In [ ]:
raw_text = load_pdf_text(PDF_PATH)

print("Кількість символів:", len(raw_text))
print(raw_text[:1000])

## 3. Очищення та розбиття на chunks

RAG не передає весь документ у LLM. Спочатку документ розбивається на невеликі фрагменти.

У цій практиці використайте простий підхід:

- `chunk_size` - приблизна кількість символів у chunk;
- `overlap` - скільки символів повторюється між сусідніми chunks;
- overlap потрібний, щоб важлива фраза не загубилася на межі двох chunks.

In [ ]:
def normalize_text(text):
    """Прибирає зайві пробіли, але не змінює зміст документа."""
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_text(text, chunk_size=900, overlap=150):
    """Розбиває текст на chunks із перекриттям."""
    text = normalize_text(text)

    # TODO: створіть список chunks
    # TODO: використайте цикл while зі змінними start та end
    # TODO: після кожного chunk зсувайте start на chunk_size - overlap
    # TODO: не додавайте порожні chunks

    raise NotImplementedError("Заповніть функцію split_text")

**Очікуваний результат:** функція `split_text(...)` повертає список непорожніх текстових фрагментів із перекриттям між сусідніми chunks.

У наступній комірці має бути видно:
- `Number of chunks` більше за 0;
- `First chunk length` не більше за переданий `chunk_size`;
- початок першого chunk як звичайний текст із PDF.

In [ ]:
chunks = split_text(raw_text, chunk_size=700, overlap=120)

print("Кількість chunks:", len(chunks))
for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk[:500])

## 4. Embeddings для chunks

Модель `SentenceTransformer` перетворює кожен chunk на вектор. Схожі за змістом тексти мають близькі вектори.

Для українських та англійських запитів використаємо multilingual-модель:

```python
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

Заповніть клітинку так, щоб отримати матрицю `chunk_embeddings` розміру:

```text
number_of_chunks x embedding_dimension
```

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

# TODO: закодуйте chunks у chunk_embeddings
# Підказка: використайте model.encode(..., normalize_embeddings=True)
# normalize_embeddings=True дозволяє використовувати dot product як cosine similarity.

chunk_embeddings = None

print("Embeddings shape:", None if chunk_embeddings is None else chunk_embeddings.shape)

**Очікуваний результат:** `chunk_embeddings` містить вектор для кожного chunk.

Після запуску комірки має з’явитися форма на кшталт `(кількість_chunks, розмірність_вектора)`, де перше число дорівнює `len(chunks)`, а не `None`.

## 5. Semantic search через cosine similarity

Тепер реалізуйте пошук релевантних chunks.

Алгоритм:

1. закодувати питання в `query_embedding`;
2. порахувати `scores = query_embedding @ chunk_embeddings.T`;
3. відсортувати індекси за спаданням score;
4. повернути top-k chunks у вигляді таблиці.

In [ ]:
def retrieve(query, chunks, chunk_embeddings, model, top_k=3):
    """Повертає top_k найбільш релевантних chunks для питання."""

    # TODO: закодуйте query через model.encode
    # TODO: порахуйте scores через dot product
    # TODO: знайдіть індекси top_k найкращих chunks
    # TODO: поверніть pandas DataFrame з колонками: rank, chunk_id, score, chunk

    raise NotImplementedError("Заповніть функцію retrieve")

**Очікуваний результат:** функція `retrieve(...)` повертає `pandas.DataFrame` з top-k найбільш релевантними chunks.

Для тестового питання наступна комірка має показати таблицю з 3 рядками і колонками `rank`, `chunk_id`, `score`, `chunk`. Значення `rank` мають іти від 1 до 3, а найрелевантніший chunk має бути першим.

In [ ]:
test_query = "What is the maximum washing temperature?"
results = retrieve(test_query, chunks, chunk_embeddings, model, top_k=3)
results

## 6. Формування контексту для LLM

LLM має відповідати не з пам'яті, а з контексту, який знайшов retrieval.

Заповніть `build_context`, щоб перетворити top chunks на один текстовий блок. Добре додати номер chunk і score: так легше перевіряти, звідки взята відповідь.

In [ ]:
def build_context(results_df):
    """Перетворює таблицю top chunks на текстовий context block."""

    # TODO: пройдіться по рядках results_df
    # TODO: для кожного chunk додайте заголовок з rank, chunk_id і score
    # TODO: об'єднайте все в один рядок через подвійний перенос рядка

    raise NotImplementedError("Заповніть функцію build_context")


def build_rag_prompt(question, context):
    return f"""
You are a careful assistant that answers questions using only the provided context.
If the answer is not present in the context, say that the document does not contain enough information.
Answer briefly and mention which chunk numbers support the answer.

Context:
{context}

Question:
{question}

Answer:
""".strip()

**Очікуваний результат:** `build_context(results)` перетворює таблицю знайдених chunks на один текстовий блок для prompt.

У наступній комірці має вивестися контекст, де кожен фрагмент має заголовок із `rank`, `chunk_id` і `score`, а між фрагментами є порожній рядок.

In [ ]:
context = build_context(results)
prompt = build_rag_prompt(test_query, context)

print(prompt[:2000])

## 7. Додавання LLM через Gemini API

На занятті використовуємо Gemini API.

Можна обрати одну з моделей:

- `gemini-2.5-flash` - ліміт приблизно 10 запитів на хвилину і 250 запитів на день;
- `gemini-2.5-flash-lite` - ліміт приблизно 30 запитів на хвилину і 1000 запитів на день.

Для практики зручно почати з `gemini-2.5-flash-lite`, а за потреби замінити модель на `gemini-2.5-flash`.

Головне правило: модель отримує **question + retrieved context**, а не весь PDF.

In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = "gemini-2.5-flash-lite"  # або "gemini-2.5-flash"
USE_LLM = GEMINI_API_KEY is not None


def call_llm(prompt):
    """Повертає відповідь Gemini для готового RAG prompt."""

    if not USE_LLM:
        return (
            "Gemini API вимкнено. Додайте GEMINI_API_KEY у змінні середовища "
            "або прямо в notebook, потім увімкніть USE_LLM=True."
        )

    # TODO: сформуйте URL у форматі:
    # https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}
    # TODO: сформуйте payload: {"contents": [{"parts": [{"text": prompt}]}]}
    # TODO: зробіть POST-запит через requests.post(...)
    # TODO: дістаньте текст із response.json()["candidates"][0]["content"]["parts"][0]["text"]

    raise NotImplementedError("Додайте виклик Gemini API у call_llm")

**Очікуваний результат:** якщо `GEMINI_API_KEY` заданий, `call_llm(prompt)` надсилає prompt у Gemini API і повертає текст відповіді моделі.

Якщо ключ не заданий, функція має повернути зрозуміле повідомлення, що Gemini API вимкнено, без падіння notebook.

In [ ]:
# Приклад прямого Gemini API-запиту без RAG.
# Використайте його для перевірки ключа перед інтеграцією в call_llm.
#
# GEMINI_API_KEY = "ВАШ_API_КЛЮЧ"
# GEMINI_MODEL = "gemini-2.5-flash-lite"
# url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"
# payload = {
#     "contents": [
#         {"parts": [{"text": "Напиши один короткий рядок про переваги вивчення програмування."}]}
#     ]
# }
# response = requests.post(url, json=payload, timeout=60)
# response.raise_for_status()
# data = response.json()
# print(data["candidates"][0]["content"]["parts"][0]["text"])

## 8. Фінальна функція `ask_pdf`

Тепер зберіть усі частини pipeline в одну функцію.

In [ ]:
def ask_pdf(question, top_k=3):
    """Повний RAG pipeline для одного питання."""

    # TODO: знайдіть top_k chunks через retrieve
    # TODO: побудуйте context через build_context
    # TODO: побудуйте prompt через build_rag_prompt
    # TODO: отримайте answer через call_llm
    # TODO: поверніть словник з question, answer, results і prompt

    raise NotImplementedError("Зберіть pipeline у функції ask_pdf")

**Очікуваний результат:** `ask_pdf(question, top_k=3)` запускає весь RAG pipeline для одного питання.

Функція має повернути словник з ключами `question`, `answer`, `results` і `prompt`. У наступній комірці має відобразитися відповідь, таблиця top chunks та сформований prompt без помилки `NotImplementedError`.

In [ ]:
answer = ask_pdf("How do I cancel delayed start?", top_k=3)

print("Question:", answer["question"])
print("\nAnswer:\n", answer["answer"])
print("\nRetrieved chunks:")
display(answer["results"][["rank", "chunk_id", "score"]])

## 9. Перевірка якості на 5 питаннях

Перевірте не тільки «легкі» питання, а й питання, відповіді на які немає в документі. Хороший RAG-асистент має вміти сказати, що контексту недостатньо.

In [ ]:
questions = [
    "What is the maximum washing temperature?",
    "How do I cancel delayed start?",
    "How often should I clean the drain filter?",
    "What does error E21 mean?",
    "How can I cancel my subscription?",  # цього немає в тестовому PDF
]

rows = []
for question in questions:
    result = ask_pdf(question, top_k=3)
    rows.append({
        "question": question,
        "answer": result["answer"],
        "top_score": float(result["results"].iloc[0]["score"]),
        "top_chunk_id": int(result["results"].iloc[0]["chunk_id"]),
    })

quality_df = pd.DataFrame(rows)
quality_df

## 10. Висновок

Заповніть короткий висновок після тестування.

Дайте відповіді:

1. На які питання система відповіла добре?
2. Де retrieval знайшов слабкий або неправильний контекст?
3. Чи відмовилась модель відповідати, коли інформації в PDF не було?
4. Що можна покращити: `chunk_size`, `overlap`, prompt, кількість `top_k`, модель embeddings?

TODO: напишіть висновок у markdown-клітинці.

